In [0]:
import sys
import os
from datetime import datetime
from pyspark.sql.functions import col, date_format

# --------------------------------------------------
# 1. Widgets (Job Parameters)
# --------------------------------------------------
# dbutils.widgets.text("customer_id", "")
dbutils.widgets.text("object_name", "")
dbutils.widgets.text("source_system", "salesforce")
dbutils.widgets.text("sf_catalog_table", "")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
sf_catalog_table = dbutils.widgets.get("sf_catalog_table")
load_type = "historical"

# --------------------------------------------------
# 2. Import Configuration dynamically
# --------------------------------------------------
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
abs_project_root = os.path.abspath(project_root)

if abs_project_root not in sys.path:
    sys.path.append(abs_project_root)

from utils.config import DataLakeConfig

# Instantiate the config
config = DataLakeConfig(source_system, customer_id, object_name, load_type)

# --------------------------------------------------
# 3. Validation & Setup
# --------------------------------------------------
if customer_id == "" or object_name == "" or sf_catalog_table == "":
    raise Exception("Missing required job parameters: customer_id, object_name, or sf_catalog_table")

# Performance config
spark.conf.set("spark.sql.shuffle.partitions", 4)

# Generate S3 Path using Config Helper
now = datetime.now()
historical_s3_path = config.get_landing_zone_timestamped_path(source_system, customer_id, object_name, load_type, now)

print("--------------------------------------------------")
print(f"🚀 Starting Historical Load")
print(f"Customer      : {customer_id}")
print(f"Object        : {object_name}")
print(f"Source        : {source_system}")
print(f"Catalog Table : {sf_catalog_table}")
print(f"S3 Path       : {historical_s3_path}")
print("--------------------------------------------------")

try:
    # 1. Fetch Max SystemModstamp for Watermark
    max_ts_df = spark.sql(f"SELECT MAX(SystemModstamp) as max_ts FROM {sf_catalog_table}")
    max_ts = max_ts_df.first()[0]

    if max_ts is None:
        print("No records found in source table.")
        dbutils.notebook.exit("0")

    print(f"📍 Max SystemModstamp found: {max_ts}")

    # 2. Read from source
    df_raw = spark.table(sf_catalog_table)
    
    # Format the Timestamp to exact String format 
    df_formatted = df_raw.withColumn("SystemModstamp", date_format(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
    
    # 3. Write to S3 
    (
        df_formatted
        .repartition(4)
        .write
        .mode("append")
        .format("parquet")
        .option("compression", "snappy")
        .save(historical_s3_path)
    )

    print(f"✅ FINISHED! Successfully landed formatted raw records to S3")

    # 4. Initialize Watermark Table (Creates if it doesn't exist)
    print("Updating watermark table safely using MERGE...")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
            source_system STRING,
            object_name STRING,
            last_processed_timestamp TIMESTAMP
        ) USING DELTA
    """)
    
    # Use MERGE INTO to avoid concurrent overwrite errors
    spark.sql(f"""
        MERGE INTO ingestion_metadata.watermarks AS target
        USING (
            SELECT '{source_system}' AS source_system, 
                   '{object_name}' AS object_name, 
                   TIMESTAMP('{max_ts}') - INTERVAL 1 MINUTE AS last_processed_timestamp
        ) AS source
        ON target.source_system = source.source_system AND target.object_name = source.object_name
        WHEN MATCHED THEN
            UPDATE SET target.last_processed_timestamp = source.last_processed_timestamp
        WHEN NOT MATCHED THEN
            INSERT (source_system, object_name, last_processed_timestamp)
            VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
    """)
    
    print(f"✅ Watermark updated safely (minus 1 minute).")



except Exception as e:
    print(f"❌ Pipeline Failed: {str(e)}")
    raise e